In [5]:
from google.cloud import bigquery
import os

def init_bigquery_client() -> bigquery.Client:
    """Initialize BigQuery client"""
    bq_credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
    project_id = os.getenv('GCP_PROJECT_ID')
    
    if not project_id:
        raise ValueError("GCP_PROJECT_ID environment variable not set")
    
    if bq_credentials_path and os.path.exists(bq_credentials_path):
        os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = bq_credentials_path
        print(f"Using BigQuery credentials: {bq_credentials_path}")
    
    return bigquery.Client(project=project_id)

#pulling data from bigquery tables
nodes_query = "SELECT * FROM `etl-testing-478716.instagram_graph.nodes`"
edges_query = "SELECT * FROM `etl-testing-478716.instagram_graph.edges`"
client = init_bigquery_client()
nodes_bq = client.query(nodes_query).to_dataframe()
edges_bq = client.query(edges_query).to_dataframe()

Using BigQuery credentials: etl-testing-478716-c0b6c2c512e0.json


/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [28]:
#deduplicating nodes based on username and full_name
nodes_dedup = nodes_bq.drop_duplicates(subset = ["username", "full_name"])

#filter to public accounts only
nodes_dedup_public = nodes_dedup[nodes_dedup['is_private'] == False].reset_index(drop=True)

#getting lockett followers
lockett_followers_list = edges_bq[edges_bq['target'] == 'locketfuxx']['source'].unique().tolist()

nodes_dedup_public['is_lockett_follower'] = nodes_dedup_public['username'].apply(lambda x: x in lockett_followers_list)

nodes_locket_follower = nodes_dedup_public[nodes_dedup_public['is_lockett_follower'] == True].reset_index(drop=True)

In [6]:
#deduplicating nodes based on username and full_name
nodes_dedup = nodes_bq.drop_duplicates(subset = ["username", "full_name"])

#filter to public accounts only
nodes_dedup_public = nodes_dedup[nodes_dedup['is_private'] == False].reset_index(drop=True)
#split into two df's based on is_verified
nodes_verified = nodes_dedup_public[nodes_dedup_public['is_verified'] == True].reset_index(drop=True)
nodes_unverified = nodes_dedup_public[nodes_dedup_public['is_verified'] == False].reset_index(drop=True)

print(f"Total nodes: {len(nodes_dedup)}")
print(f"Verified nodes: {len(nodes_verified)}")
print(f"Unverified nodes: {len(nodes_unverified)}")

Total nodes: 5290
Verified nodes: 542
Unverified nodes: 2810


In [29]:
from apify_client import ApifyClient
from main import run_profile_crawl

In [30]:
import time
import pandas as pd

In [31]:
print("Running the next step...")
username_list = nodes_locket_follower['username'].tolist()
print(f"Total usernames to crawl: {len(username_list)}")

Running the next step...
Total usernames to crawl: 1546


In [33]:
profiles = run_profile_crawl(username_list)

#writing profiles to parquet file
run_id = str(int(time.time()))
profiles_df = pd.DataFrame(profiles)
profiles_df.to_parquet(f"profiles_{run_id}.parquet", index=False)

Crawling profiles for 1546 accounts...


[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> Status: RUNNING, Message: 
[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> 2026-03-31T20:50:27.030Z ACTOR: Pulling container image of build AIf7xvwjg39uyRUcq from registry.
[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> 2026-03-31T20:50:27.032Z ACTOR: Creating container.
[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> 2026-03-31T20:50:27.082Z ACTOR: Starting container.
[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> 2026-03-31T20:50:27.083Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> 2026-03-31T20:50:27.621Z INFO  System info {"apifyVersion":"3.6.0","apifyClientVersion":"2.22.2","crawleeVersion":"3.16.0","osType":"Linux","nodeVersion":"v22.22.1"}
[apify.instagram-profile-scraper runId:dLG6Gb6png5yqUFHt] -> 2026-03-31T20:50:27.745Z INFO  Results Limit [object Object], ACTOR_MAX_PAID_DATASET_ITEMS
[

Fetched profiles for 1546 accounts


In [ ]:

# Turning profiles into dataframe and saving as parquet file
profiles_df['Ratio'] = profiles_df['followersCount'] / profiles_df['followsCount'].replace(0, 1)  # Avoid division by zero
# run_id = str(int(time.time()))
# profiles_df.to_parquet(f"profiles_{run_id}.parquet", index=False)
# print(f"Saved profiles with ratio as parquet file locally")

print("Top 20 profiles by ratio and total followers:")
top_profiles = profiles_df.sort_values(by=["Ratio", "followersCount"], ascending=False).head(20)


Top 20 profiles by ratio and total followers:
                    username  followersCount  followsCount       Ratio  \
298              raicestexas        148685.0         530.0  280.537736   
288                 zelladay        166025.0        1841.0   90.181966   
448              thetessalee         55773.0         760.0   73.385526   
1501               sad.jazzy         29695.0         433.0   68.579677   
673               redrumclub         53070.0         832.0   63.786058   
1263                 kgsratx         27030.0         437.0   61.853547   
1328         almondmilkhunni        119152.0        2107.0   56.550546   
891              maysterling         35430.0         694.0   51.051873   
141         _mechellenicole_         70746.0        1473.0   48.028513   
1267               imtomfurd         64143.0        1559.0   41.143682   
496            kingdomaustin         29968.0         847.0   35.381346   
131          verylonely.shop         24166.0         753.0   32.09

In [48]:

keep_cols = ['inputUrl', 'id', 'username', 'url', 'fullName', 'biography',
       'externalUrls', 'followersCount', 'followsCount', 'hasChannel',
       'highlightReelCount', 'isBusinessAccount', 'joinedRecently',
       'businessCategoryName', 'private', 'verified', 'profilePicUrl',
       'profilePicUrlHD', 'igtvVideoCount', 'relatedProfiles',
       'latestIgtvVideos', 'postsCount', 'latestPosts', 'fbid', 'externalUrl',
       'externalUrlShimmed','Ratio']

profiles_df = profiles_df[keep_cols]

# repeated columns 
for col in ["externalUrls", "relatedProfiles", "latestIgtvVideos", "latestPosts"]:
    if col in profiles_df.columns:
        profiles_df[col] = profiles_df[col].apply(lambda x: x if isinstance(x, list) else [])
        
#creating profile table with username, followersCount, followsCount, and Ratio
from main import BQ_DATASET, GCP_PROJECT

bq = init_bigquery_client()
    
# Create dataset if it doesn't exist
dataset_id = f"{GCP_PROJECT}.{BQ_DATASET}"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"
bq.create_dataset(dataset, exists_ok=True)
print(f"Dataset {BQ_DATASET} ready")

# Define nodes table schema
profiles_schema = [
    bigquery.SchemaField("inputUrl", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("id", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("username", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("url", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("fullName", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("biography", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("externalUrls", "STRING", mode="REPEATED"),
    bigquery.SchemaField("followersCount", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("followsCount", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("hasChannel", "BOOLEAN", mode="NULLABLE"),
    bigquery.SchemaField("highlightReelCount", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("isBusinessAccount", "BOOLEAN", mode="NULLABLE"),
    bigquery.SchemaField("joinedRecently", "BOOLEAN", mode="NULLABLE"),
    bigquery.SchemaField("businessCategoryName", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("private", "BOOLEAN", mode="NULLABLE"),
    bigquery.SchemaField("verified", "BOOLEAN", mode="NULLABLE"),
    bigquery.SchemaField("profilePicUrl", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("profilePicUrlHD", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("igtvVideoCount", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("relatedProfiles", "STRING", mode="REPEATED"),
    bigquery.SchemaField("latestIgtvVideos", "STRING", mode="REPEATED"),
    bigquery.SchemaField("postsCount", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("latestPosts", "STRING", mode="REPEATED"),
    bigquery.SchemaField("fbid", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("externalUrl", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("externalUrlShimmed", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("Ratio", "FLOAT", mode="NULLABLE"),
]


def ensure_required_columns(df: pd.DataFrame, schema_function) -> pd.DataFrame:
    """
    Ensure all required columns exist in DataFrame based on BigQuery schema definition.
    
    Args:
        df: Input DataFrame
        schema_function: Function that returns BigQuery schema (e.g., define_schema_users)
    
    Returns:
        DataFrame with all required columns from schema
    """
    # Get schema from the schema function
    schema = schema_function()
    
    # Extract column names and types from schema
    for field in schema:
        col_name = field.name
        field_type = field.field_type
        
        if col_name not in df.columns:
            # Set appropriate defaults based on BigQuery field type
            if field_type == 'BOOLEAN':
                df[col_name] = False
            elif field_type in ['INTEGER', 'FLOAT', 'NUMERIC']:
                df[col_name] = None
            elif field_type in ['TIMESTAMP', 'DATETIME', 'DATE']:
                df[col_name] = None
            else:  # STRING and other types
                df[col_name] = None
    
    # Fill NaN values for boolean columns
    boolean_fields = [field.name for field in schema if field.field_type == 'BOOLEAN']
    for col in boolean_fields:
        if col in df.columns:
            df.loc[:, col] = df[col].fillna(False)
    
    # Return DataFrame with only schema columns in the right order
    schema_columns = [field.name for field in schema]
    available_columns = [col for col in schema_columns if col in df.columns]
    
    # Create a clean copy with reset index to avoid PyArrow conversion issues
    result_df = df[available_columns].copy()
    result_df.reset_index(drop=True, inplace=True)

    # Ensure repeated fields are lists of strings
    repeated_fields = ["externalUrls", "relatedProfiles", "latestIgtvVideos", "latestPosts"]
    for col in repeated_fields:
        if col in result_df.columns:
            result_df[col] = result_df[col].apply(
                lambda x: [str(i) for i in x] if isinstance(x, list) else ([] if pd.isna(x) or x is None else [str(x)])
            )
    
    return result_df

# Ensure all required columns exist in profiles_df based on profiles_schema
profiles_df = ensure_required_columns(profiles_df, lambda: profiles_schema)

/var/folders/d6/wbxdxj_s71ngq7ft6m6_6krc0000gn/T/ipykernel_18182/1272207864.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  profiles_df[col] = profiles_df[col].apply(lambda x: x if isinstance(x, list) else [])


Using BigQuery credentials: etl-testing-478716-c0b6c2c512e0.json
Dataset instagram_graph ready


/var/folders/d6/wbxdxj_s71ngq7ft6m6_6krc0000gn/T/ipykernel_18182/1272207864.py:94: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.loc[:, col] = df[col].fillna(False)


In [49]:


# Create profiles table if it doesn't exist
profiles_table = bigquery.Table(f"{dataset_id}.profiles", schema=profiles_schema)
bq.create_table(profiles_table, exists_ok=True)

# Insert profiles data into BigQuery
job_config = bigquery.LoadJobConfig(
    schema=profiles_schema,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
load_job = bq.load_table_from_dataframe(profiles_df, f"{dataset_id}.profiles", job_config=job_config)
load_job.result()  # Wait for the job to complete
print(f"Inserted {len(profiles_df)} profiles into {dataset_id}.profiles")

/opt/miniconda3/envs/heyyall/lib/python3.13/site-packages/google/cloud/bigquery/_pandas_helpers.py:484: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Inserted 1546 profiles into etl-testing-478716.instagram_graph.profiles


In [37]:

top_profiles[["username", "followersCount", "followsCount", "Ratio", "isBusinessAccount", "private", "businessCategoryName"]]

,username,followersCount,followsCount,Ratio,isBusinessAccount,private,businessCategoryName
298,raicestexas,148685.0,530.0,280.537736,True,False,"None,Nonprofit organization"
288,zelladay,166025.0,1841.0,90.181966,True,False,"None,Artist"
448,thetessalee,55773.0,760.0,73.385526,False,False,Model
1501,sad.jazzy,29695.0,433.0,68.579677,True,False,"None,Artist"
673,redrumclub,53070.0,832.0,63.786058,True,False,"None,Musician/band"
1263,kgsratx,27030.0,437.0,61.853547,False,False,None
1328,almondmilkhunni,119152.0,2107.0,56.550546,False,False,Musician/band
891,maysterling,35430.0,694.0,51.051873,False,False,None
141,_mechellenicole_,70746.0,1473.0,48.028513,False,False,Health/beauty
1267,imtomfurd,64143.0,1559.0,41.143682,False,False,None


In [39]:
top_profiles['followersCount'].sum() / 1000

np.float64(1048.382)

In [ ]:
#running crawl on selected accounts 
from main import store_to_bigquery
from main import run_bfs_crawl

seed_accounts = []


run_id = str(int(time.time()))
G = run_bfs_crawl(seed_accounts)
#before storing save as parquet files and save locally as backup 
# Insert data
nodes = [{"username": n, **d} for n, d in G.nodes(data=True)]
edges = [{"source": u, "target": v, **d} for u, v, d in G.edges(data=True)]

# Save as parquet files locally
nodes_df = pd.DataFrame(nodes)
edges_df = pd.DataFrame(edges)
nodes_df.to_parquet(f"nodes_{run_id}.parquet", index=False)
edges_df.to_parquet(f"edges_{run_id}.parquet", index=False)
print(f"Saved nodes and edges as parquet files locally")

#saving to biquery dataset
store_to_bigquery(G)

In [18]:
#visualizing network 
import networkx as nx

G = nx.DiGraph()  # Directed graph for following relationships
# Add nodes and edges to G based on your data

nodes_bq = nodes_bq.drop_duplicates(subset=['username']).reset_index(drop=True)
edges_bq = edges_bq.drop_duplicates(subset=['source', 'target']).reset_index(drop=True)

for item,row in nodes_bq.iterrows():
    G.add_node(row['username'], is_verified=row['is_verified'], is_private=row['is_private'])

for item,row in edges_bq.iterrows():
    G.add_edge(row['source'], row['target'], **row.to_dict())
import pandas as pd

# Replace pd.NA and None with empty string for all node attributes
for n, d in G.nodes(data=True):
    for k, v in list(d.items()):
        if v is pd.NA or v is None:
            d[k] = ""

# Replace pd.NA and None with empty string for all edge attributes
for u, v, d in G.edges(data=True):
    for k, v2 in list(d.items()):
        if v2 is pd.NA or v2 is None:
            d[k] = ""

nx.write_gexf(G, "my_graph.gexf")

In [60]:
#using skip trace apify api 

from crawler import APIFY_TOKEN



def fetch_skip_trace(input_params: dict) -> list[dict]:
    """
    For each name, phone number, or email, fetch associated Instagram accounts using Skip Trace API.
    At least one of name, phone_number, or email must be provided.
    """
    # if not (input_params.get("name") or input_params.get("phone_number") or input_params.get("email")):
    #     raise ValueError("At least one of name, phone_number, or email must be provided.")
    
    POSH_TRACE = 'hypebridge/posh-vip'
    apify = ApifyClient(APIFY_TOKEN)

    run = apify.actor(POSH_TRACE).call(run_input=input_params)
    return list(apify.dataset(run["defaultDatasetId"]).iterate_items())



In [86]:
test_posh = {"startUrls": [
    {"url": "https://posh.vip/explore?when=This+Month&sort=Largest&location=%7B%22type%22%3A%22custom%22%2C%22location%22%3A%22Austin%2C+TX%2C+USA%22%2C%22long%22%3A-97.7430608%2C%22lat%22%3A30.267153%7D"}
],
"maxEvents": 500,
"scrapeEventDetails": True
}

big_results_df = fetch_skip_trace(test_posh)
#store as parquet file locally
run_id = str(int(time.time()))
big_results_df = pd.DataFrame(big_results_df)
big_results_df.to_parquet(f"posh_results_{run_id}.parquet", index=False)
print(f"Saved Posh results as parquet file locally")

[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> Status: RUNNING, Message: 
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:22.998Z ACTOR: Pulling container image of build VGlEYq35ZQlbL044v from registry.
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:24.137Z ACTOR: Creating container.
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:24.188Z ACTOR: Starting container.
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:24.189Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:26.589Z INFO  System info {"apifyVersion":"3.5.2","apifyClientVersion":"2.21.0","crawleeVersion":"3.15.3","osType":"Linux","nodeVersion":"v20.19.6"}
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:26.678Z INFO  Starting Posh.vip scraper {"startUrlCount":1,"scrapeEventDetails":true,"maxEvents":500}
[apify.posh-vip runId:SuZE3Qpfs9I3OEuNU] -> 2026-04-01T16:50:26.680Z INFO  Detected deeplink

Saved Posh results as parquet file locally


In [87]:
#Getting event level results
posh_results_df = big_results_df.copy()
posh_results_df = posh_results_df.sort_values(by=["attendingCount"], ascending=False)

#Getting organizer level resutls
organizer_df = posh_results_df[["organizerName", "organizer", "organizerId"]].drop_duplicates(subset=['organizerId']).reset_index(drop=True)

#number of events per organizer
num_events_df = posh_results_df.groupby(["organizerId"]).agg(num_events = ('organizerId', 'count'))

# Expand 'data' column into new columns
new_cols = pd.json_normalize(organizer_df['organizer'])
#creating final organizer df
organizer_df_final = pd.concat([organizer_df.drop(columns=['organizer']), new_cols.drop(columns=['id','logo','currency','country','name'])], axis=1)
organizer_df_final = organizer_df_final.merge(num_events_df, on="organizerId", how="left")

In [105]:
# creating list of organizer urls to crawl
list_of_organizer_urls = organizer_df_final['url'].tolist()

In [107]:
import re

def add_g_to_posh_links(links):
    return [re.sub(r"(https://posh.vip/)([^/]+)$", r"\1g/\2", link) for link in links]


converted_urls = add_g_to_posh_links(list_of_organizer_urls)
print(converted_urls)

# Assuming converted_urls is already defined as your list of URLs
url_dicts = [{"url": url} for url in converted_urls]
print(len(url_dicts))

['https://posh.vip/g/secretsociety', 'https://posh.vip/g/ILOV3RNB', 'https://posh.vip/g/malavida', 'https://posh.vip/g/only-the-crew-llc', 'https://posh.vip/g/eastendballroom', 'https://posh.vip/g/twins-night-club', 'https://posh.vip/g/artchives', 'https://posh.vip/g/secretdiscosociety', 'https://posh.vip/g/where-yall-at-though', 'https://posh.vip/g/afroexpress-1', 'https://posh.vip/g/perreo-club', 'https://posh.vip/g/riviere-austin', 'https://posh.vip/g/aerolifeshows', 'https://posh.vip/g/sumn2say', 'https://posh.vip/g/nidasnexus', 'https://posh.vip/g/underground-faith', 'https://posh.vip/g/txst-nupes', 'https://posh.vip/g/lovemorernb', 'https://posh.vip/g/chas-and-champ', 'https://posh.vip/g/legvcy-events', 'https://posh.vip/g/events-by-samantha-2', 'https://posh.vip/g/punchline-at-the-pole', 'https://posh.vip/g/lez-in-the-capital', 'https://posh.vip/g/chunk-bros', 'https://posh.vip/g/SoulBloom', 'https://posh.vip/g/yuli', 'https://posh.vip/g/amplify-edm', 'https://posh.vip/g/events-

In [111]:
#chunking urls into groups of 10 to avoid overloading the system
chunk_size = 10
all_results = []

for i in range(0, len(converted_urls), chunk_size):
    # Get the current chunk
    chunk_urls = converted_urls[i:i+chunk_size]
    url_dicts = [{"url": url} for url in chunk_urls]
    
    test_posh = {
        "startUrls": url_dicts,
        "maxEvents": 2,
        "scrapeEventDetails": True
    }
    
    group_pulls = fetch_skip_trace(test_posh)
    all_results.extend(group_pulls)
    print(f"Processed chunk {i//chunk_size + 1}, got {len(group_pulls)} results")
    
#store as parquet file locally
run_id = str(int(time.time()))
group_pulls_df = pd.DataFrame(all_results)
group_pulls_df.to_parquet(f"posh_group_results_{run_id}.parquet", index=False)
print(f"Saved {len(all_results)} total Posh results as parquet file locally")

[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> Status: RUNNING, Message: 
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:07.768Z ACTOR: Pulling container image of build VGlEYq35ZQlbL044v from registry.
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:07.770Z ACTOR: Creating container.
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:07.825Z ACTOR: Starting container.
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:07.827Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:09.761Z INFO  System info {"apifyVersion":"3.5.2","apifyClientVersion":"2.21.0","crawleeVersion":"3.15.3","osType":"Linux","nodeVersion":"v20.19.6"}
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:09.913Z INFO  Starting Posh.vip scraper {"startUrlCount":10,"scrapeEventDetails":true,"maxEvents":2}
[apify.posh-vip runId:CgB7HodsvgS7cMvd1] -> 2026-04-01T17:21:09.915Z INFO  Detected group pag

Processed chunk 1, got 18 results


[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> Status: RUNNING, Message: 
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:21.324Z ACTOR: Pulling container image of build VGlEYq35ZQlbL044v from registry.
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:21.326Z ACTOR: Creating container.
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:21.414Z ACTOR: Starting container.
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:21.415Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:22.544Z INFO  System info {"apifyVersion":"3.5.2","apifyClientVersion":"2.21.0","crawleeVersion":"3.15.3","osType":"Linux","nodeVersion":"v20.19.6"}
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:22.623Z INFO  Starting Posh.vip scraper {"startUrlCount":10,"scrapeEventDetails":true,"maxEvents":2}
[apify.posh-vip runId:Z7njdGkGuhzqqpRt6] -> 2026-04-01T17:21:22.624Z INFO  Detected group pag

Processed chunk 2, got 17 results


[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> Status: RUNNING, Message: 
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:32.999Z ACTOR: Pulling container image of build VGlEYq35ZQlbL044v from registry.
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:33.001Z ACTOR: Creating container.
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:33.064Z ACTOR: Starting container.
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:33.065Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:34.414Z INFO  System info {"apifyVersion":"3.5.2","apifyClientVersion":"2.21.0","crawleeVersion":"3.15.3","osType":"Linux","nodeVersion":"v20.19.6"}
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:34.511Z INFO  Starting Posh.vip scraper {"startUrlCount":10,"scrapeEventDetails":true,"maxEvents":2}
[apify.posh-vip runId:oo8Ue3OM4BNCZvuqU] -> 2026-04-01T17:21:34.512Z INFO  Detected group pag

...

Processed chunk 9, got 18 results


[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> Status: RUNNING, Message: 
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:14.699Z ACTOR: Pulling container image of build VGlEYq35ZQlbL044v from registry.
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:14.701Z ACTOR: Creating container.
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:14.889Z ACTOR: Starting container.
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:14.889Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:17.313Z INFO  System info {"apifyVersion":"3.5.2","apifyClientVersion":"2.21.0","crawleeVersion":"3.15.3","osType":"Linux","nodeVersion":"v20.19.6"}
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:17.412Z INFO  Starting Posh.vip scraper {"startUrlCount":10,"scrapeEventDetails":true,"maxEvents":2}
[apify.posh-vip runId:24JBwdERjQzhl8KQi] -> 2026-04-01T17:23:17.413Z INFO  Detected group pag

Processed chunk 10, got 18 results


[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> Status: RUNNING, Message: 
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:27.458Z ACTOR: Pulling container image of build VGlEYq35ZQlbL044v from registry.
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:27.460Z ACTOR: Creating container.
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:27.517Z ACTOR: Starting container.
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:27.517Z ACTOR: Running under "LIMITED_PERMISSIONS" permission level.
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:29.485Z INFO  System info {"apifyVersion":"3.5.2","apifyClientVersion":"2.21.0","crawleeVersion":"3.15.3","osType":"Linux","nodeVersion":"v20.19.6"}
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:29.578Z INFO  Starting Posh.vip scraper {"startUrlCount":4,"scrapeEventDetails":true,"maxEvents":2}
[apify.posh-vip runId:3UwyhK1FyHVJ4MYcX] -> 2026-04-01T17:23:29.579Z INFO  Detected group page

Processed chunk 11, got 8 results
Saved 188 total Posh results as parquet file locally


In [115]:
group_pulls_df[group_pulls_df['type']=='group'].sort_values(by=["totalAttendees"], ascending=False)

,type,groupId,groupUrl,groupName,bio,avatarUrl,contactEmail,website,socials,country,...,activityEnabled,displayOnThirdPartySites,isPaymentPlanEligible,tablesCount,fees,song,youtubeLink,videos,accentColor,lightmode
7,group,65b8ab2753b2972dfae5ea23,https://posh.vip/g/secretsociety,Events by secret society,elevating experiences since 22',https://posh-images-alts-production.s3.amazona...,None,None,{'instagram': 'https://www.instagram.com/secre...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,group,6297bbe528fb980033b2401d,https://posh.vip/g/secretdiscosociety,Secret Disco Society,Secret Disco Society is Austin’s premier under...,https://posh-images-alts-production.s3.amazona...,secretdiscoevents@gmail.com,https://secretdiscosociety.com,{'instagram': 'https://www.instagram.com/secre...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,group,6734641693517bb6a2f3ddbf,https://posh.vip/g/lovemorernb,Love More Rnb,for the rnb lovers,https://posh-images-alts-production.s3.amazona...,info@lovemorernb.com,None,{'instagram': 'https://www.instagram.com/lovem...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41,group,6606047e4c657f5dcc91949d,https://posh.vip/g/halfdays,Halfdays,"Hello, we're Halfdays. We're on a mission to b...",https://posh-images-alts-production.s3.amazona...,marketing@halfdays.com,https://www.halfdays.com/,{'instagram': 'https://www.instagram.com/halfd...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159,group,656924e03e5f60737b21b04a,https://posh.vip/g/methodthree,M3,We host an array of in-house events to cater t...,https://posh-images-alts-production.s3.amazona...,Roxy@methodthree.com,https://methodthree.com/,{'instagram': 'https://www.instagram.com/m3atx'},US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,group,68807e9e9b513722283dbfc3,https://posh.vip/g/themove-atx,The Move,Community | VIBES x FOMO,https://posh-images-alts-production.s3.amazona...,None,None,{'instagram': 'https://www.instagram.com/sipea...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130,group,68914e52ea8e48ff8b0b7038,https://posh.vip/g/sonus-productions,Sonus Productions,Sonus Productions is a cutting-edge event prod...,https://posh-images-alts-production.s3.amazona...,onket_industries@outlook.com,https://sonusproductions.net,{'instagram': 'https://www.instagram.com/sonus...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
111,group,69ac58ddec1e269e4d97bd38,https://posh.vip/g/ruffhouse-collective,Ruffhouse Collective,None,https://posh-images-alts-production.s3.amazona...,ruffhouse512@gmail.com,None,{},US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,group,65963a60aa3436a647f480e5,https://posh.vip/g/atxsupperclub,ATX Supper Club,ATX SUPPER CLUB WAS CREATED OUT OF A LOVE FOR ...,https://images.posh.vip/images/7a49cc50-3976-4...,hello@atxsupperclub.com,https://atxsupperclub.com/,{'instagram': 'https://www.instagram.com/atx.s...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
small_mala_vida_result

[{'type': 'group',
  'groupId': '68e16fc6cf45d1389f698c2c',
  'groupUrl': 'https://posh.vip/g/malavida',
  'groupName': 'MALA VIDA',
  'bio': 'Mala Vida is a premier Latin nightclub located in the heart of Austin. Known for its vibrant energy, top-tier DJs, and unforgettable parties, Mala Vida specializes in creating exclusive VIP experiences that bring the best of Latin nightlife to Sixth Street. From bottle service to high-energy reggaetón nights, Mala Vida sets the standard for Austin’s Latin club scene.',
  'avatarUrl': 'https://posh-images-alts-production.s3.amazonaws.com/68e33b38a22c00402622e879/600x600.png',
  'contactEmail': None,
  'website': None,
  'socials': {'instagram': 'https://www.instagram.com/malavidaaustin'},
  'country': 'US',
  'currency': 'USD',
  'verified': False,
  'totalAttendees': 4458,
  'totalEvents': 31,
  'createdAt': '2025-10-04T19:04:42.018Z',
  'scrapedAt': '2026-04-01T16:42:28.373Z',
  'source': 'posh_vip'},
 {'eventId': '68e3057a747bf3df1659de7a',
  

In [82]:
pd.DataFrame(mala_vida_results)

,type,groupId,groupUrl,groupName,bio,avatarUrl,contactEmail,website,socials,country,...,activityEnabled,displayOnThirdPartySites,isPaymentPlanEligible,tablesCount,fees,song,youtubeLink,videos,accentColor,lightmode
0,group,68e16fc6cf45d1389f698c2c,https://posh.vip/g/malavida,MALA VIDA,Mala Vida is a premier Latin nightclub located...,https://posh-images-alts-production.s3.amazona...,NaN,NaN,{'instagram': 'https://www.instagram.com/malav...,US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Rakata', 'artist': 'Wisin & Yandel'}",NaN,[],#2ce6ea,False
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Gasolina', 'artist': 'Daddy Yankee'}",NaN,[],#661e1f,False
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],None,NaN,[],#99047e,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],None,NaN,[],#985978,False
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Yandel 150', 'artist': 'Yandel'}",NaN,[],#2e5c91,False
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Chambea', 'artist': 'Bad Bunny'}",NaN,[],#cc3c3c,False
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Yeah! (feat. Lil Jon & Ludacris)', '...",NaN,[],#9e1c4c,False
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Passionfruit', 'artist': 'Drake'}",NaN,[],#94541c,False
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,False,True,False,0.0,[],"{'name': 'Rakata', 'artist': 'Wisin & Yandel'}",NaN,[],#1c832c,False


In [116]:
#using skip trace apify api 

from crawler import APIFY_TOKEN



def fetch_luma(input_params: dict) -> list[dict]:
    """
    For each name, phone number, or email, fetch associated Instagram accounts using Skip Trace API.
    At least one of name, phone_number, or email must be provided.
    """
    # if not (input_params.get("name") or input_params.get("phone_number") or input_params.get("email")):
    #     raise ValueError("At least one of name, phone_number, or email must be provided.")
    
    LUMA_TRACE = 'matyascimbulka/luma-event-scraper'
    apify = ApifyClient(APIFY_TOKEN)

    run = apify.actor(LUMA_TRACE).call(run_input=input_params)
    return list(apify.dataset(run["defaultDatasetId"]).iterate_items())


In [ ]:
luma_input = {
  "slugs": [
    "ai",
    "tech"
  ],
  "cities": [
    "Austin"
  ]
}